In [1]:
import lerobot
print("LeRobot OK: ", lerobot.__version__)

LeRobot OK:  0.4.4


In [2]:
import sys
print(sys.executable)

import num2words


/home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/bin/python


In [3]:
import torchcodec
from torchcodec.decoders import VideoDecoder
print("torchcodec OK")

torchcodec OK


In [4]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors

# Swap this import per-policy
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

In [5]:
# load a policy
model_id = "lerobot/smolvla_base"  # <- swap checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)
device = torch.device(device)

Device:  cuda


In [6]:
policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...


In [7]:
import sys, importlib.util, site, os

print("Python executable:", sys.executable)
print("sys.path[0]:", sys.path[0])
print("Conda env:", os.environ.get("CONDA_PREFIX"))

spec = importlib.util.find_spec("num2words")
print("num2words spec:", spec)
if spec is not None:
    import num2words
    print("num2words module file:", getattr(num2words, "__file__", None))
    print("num2words version:", getattr(num2words, "__version__", "no __version__"))
print("site-packages:", site.getsitepackages())


Python executable: /home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/bin/python
sys.path[0]: /home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/lib/python310.zip
Conda env: /home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA
num2words spec: ModuleSpec(name='num2words', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7a5aa8584b80>, origin='/home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/lib/python3.10/site-packages/num2words/__init__.py', submodule_search_locations=['/home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/lib/python3.10/site-packages/num2words'])
num2words module file: /home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/lib/python3.10/site-packages/num2words/__init__.py
num2words version: no __version__
site-packages: ['/home/lenena-iker/miniforge3/envs/MIOTI-LeRobot-SmolVLA/lib/python3.10/site-packages']


In [8]:
# load a lerobotdataset
dataset = LeRobotDataset("lerobot/libero")

In [10]:
# pick an episode
episode_index = 0

# each episode corresponds to a contiguous range of frame indices
from_idx = dataset.meta.episodes["dataset_from_index"][episode_index]
to_idx   = dataset.meta.episodes["dataset_to_index"][episode_index]

# get a single frame from that episode (e.g. the first frame)
frame_index = from_idx
frame = dict(dataset[frame_index])

# Map dataset image keys --> model expected keys
frame["observation.images.camera1"] = frame.pop("observation.images.image")
frame["observation.images.camera2"] = frame.pop("observation.images.image2")

# Si el modelo espera 3 cámaras y solo tienes 2:
# duplica una (solo para prueba / demo)
frame["observation.images.camera3"] = frame["observation.images.camera2"]


batch = preprocess(frame)
with torch.inference_mode():
    pred_action = policy.select_action(batch)
    # use your policy postprocess, this post process the action
    # for instance unnormalize the actions, detokenize it etc..
    pred_action = postprocess(pred_action)
